# Phase 6 — Sensor / image / multimodal distance-estimation model development (small-scale validation)

Confirm forward/backward runs cleanly for all three models (sensor-only, image-only,
fusion) on a small subset in the Notebook.
**Full-scale training is delegated to Phase 7's ML Job (we don't train on the full data here).**
The architecture and hyperparameters are inherited from the 45-minute version
(`src/model.py`, `conf/train.yaml`). Unchanged, since the ablation is already known to work.

**What this notebook does, cell by cell:**

1. Import the shared `src/` library and load `conf/train.yaml`'s hyperparameters
   (cell 2).
2. Train all three modalities (sensor-only, image-only, fusion) on a **500-sample
   subset** — just enough to catch a broken forward/backward pass quickly, not
   enough to say anything about real-world accuracy (cell 4).
3. Assert each modality actually had training data and print a one-line summary
   per modality (cell 6).
4. Print each modality's MAE/R²/baseline on this tiny subset, purely for reference
   — these numbers are expected to look noisy/unreliable at this scale (cell 8).

If cells 4 and 6 both run without error, the model code is safe to hand off to
Phase 7's ML Job for full-scale (all-of-Chunk_1) distributed training.


### Cell: resolve the repo root and load config/imports

Before `from src...` can work, Python needs the repo root (the directory
containing `src/`) on `sys.path`. Whether that root is `"."` or `".."` depends on
how this notebook was deployed — a Notebook synced from a Workspace uses the
notebook file's own directory as the working directory (so the repo root is `".."`,
one level up from `notebooks/`), while other deployment paths can differ — so the
loop just checks both candidates and uses whichever one actually has a `src/`
subdirectory.

Once the path is set, it imports `get_active_session()` (the Snowpark session this
Notebook is already running in), `load_config` and `run` from the shared `src/`
library, then loads `conf/train.yaml` into `cfg` — the same hyperparameter file
Phase 7's full-scale ML Job will use later, so whatever runs (or fails) here is
representative of what Phase 7 will do at scale.


In [ ]:
import sys
from pathlib import Path

# Repo root can be the notebook's own cwd or its parent, depending on how the notebook
# was deployed (CREATE NOTEBOOK FROM stage uses the stage root as cwd; a Workspace-synced
# notebook uses the notebook's own directory as cwd) -- pick whichever actually has `src/`.
for _candidate in (".", ".."):
    if (Path(_candidate) / "src").is_dir():
        sys.path.insert(0, _candidate)
        break

from snowflake.snowpark.context import get_active_session
from src.utils import load_config
from src.train import run

session = get_active_session()
cfg = load_config("conf/train.yaml")


## Confirm forward/backward for all 3 modalities on a small subset

Restricting to a few hundred samples via `max_samples` lets this complete in seconds to
tens of seconds even in a Notebook.


### Cell: train each modality on 500 samples

Loops over the three modalities and calls `src.train.run()` for each one with
`max_samples=500` and `dry_run=False`. This is the *same* `run()` function Phase 7
calls at full scale (all ~22,500 samples) — the only difference here is the sample
cap, which keeps each modality's training loop to a handful of seconds. Each call
returns a `meta` dict (training time, best validation loss, sample counts, and
evaluation metrics) that gets stored in the `results` dict keyed by modality name,
and printed as it completes. **Expect three `=== {modality} ===` blocks** in the
output, each followed by the training progress and the returned `meta` dict — no
exceptions, and no modality silently producing 0 training samples.


In [ ]:
results = {}
for modality in ["sensor", "image", "fusion"]:
    print(f"=== {modality} ===")
    meta = run(session, cfg, modality, out_dir=f"/tmp/phase6_{modality}",
               dry_run=False, max_samples=500)
    results[modality] = meta
    print(meta)

## Acceptance criteria: all 3 modalities complete their training loop without error, and the loss decreases

(Absolute accuracy is not evaluated here — that happens during Phase 7's full-scale training.)


### Cell: assert non-empty training data and print a one-line summary

For each modality, asserts `meta["n_train"] > 0` — i.e. that the train split
(after the 500-sample cap and this small dataset's segment-level split) wasn't
accidentally empty (a real historical bug: a `max_samples` too small can, by bad
luck, draw all its rows from val/test segments and leave 0 rows in train). If the
assertion passes, it prints
`{modality}: OK (train_secs=..., best_val_loss=...)` for each of the three
modalities. **This is the actual pass/fail gate for Phase 6** — if any assertion
raises, do not proceed to Phase 7 until the root cause is fixed.


In [ ]:
for modality, meta in results.items():
    assert meta["n_train"] > 0, f"{modality}: train split is empty"
    print(f"{modality}: OK (train_secs={meta['train_secs']}, best_val_loss={meta['best_val_loss']})")

## Reference: quick metrics on this subset (production evaluation happens in Phase 7/8)


### Cell: print MAE / R² / baseline for each modality

For each modality's stored `meta`, prints just the `mae_m` (mean absolute error in
meters), `r2` (R² on this 500-sample subset's tiny test split), and
`baseline_mae_m` (the MAE of "always predict the mean distance", i.e. the number
each model needs to beat to be useful at all). **Don't read too much into these
numbers** — with only ~500 samples split across train/val/test, the test set here
is tiny (tens of rows) and R²/MAE will bounce around a lot from run to run; the
real, statistically meaningful ablation comparison happens in Phase 7 (full Chunk_1)
and is visualized in Phase 8's dashboard. This cell exists only so you can eyeball
"is fusion at least in the same ballpark as image, and clearly better than sensor"
before committing to a much longer full-scale run.


In [ ]:
for modality, meta in results.items():
    m = meta["metrics"]
    print(modality, {k: v for k, v in m.items() if k in ("mae_m", "r2", "baseline_mae_m")})


Once small-scale validation passes, submit `jobs/phase7_train_distributed.py` as an
ML Job to proceed to full-scale training on all of Chunk_1 (Phase 7) + logging to
Experiments/Registry (Phase 8).
